# Generating and customising plots

This example illustrates how to use the plotting functions with 'matplotlib' and 'plotly' as backends
and how to customise the plots.

We use Monte Carlo sampling to generate a result for plotting.

### Setting up the environment

If you don't already have PyBOP installed, check out the [installation guide](https://pybop-docs.readthedocs.io/en/latest/installation.html) first.

We begin by importing the necessary libraries. Let's also fix the random seed to generate consistent output during development.

In [ ]:
%pip install --upgrade SciencePlots -q

import numpy as np
import pybamm

import pybop

np.random.seed(8)  # users can remove this line

## Create a model, dataset and optimisation problem

First set the model and parameter values.

In [ ]:
model = pybamm.lithium_ion.SPM()
parameter_values = pybamm.ParameterValues("Chen2020")
parameter_values.set_initial_state(0.5);

Generate a synthetic dataset and define the fitting parameters.

In [ ]:
sigma = 0.005
experiment = pybamm.Experiment(["Discharge at 0.5C for 3 minutes (5 second period)"])
solution = pybamm.Simulation(
    model, parameter_values=parameter_values, experiment=experiment
).solve()
dataset = pybop.Dataset(
    {
        "Time [s]": solution.t,
        "Current [A]": solution["Current [A]"].data,
        "Voltage [V]": pybop.add_noise(solution["Voltage [V]"].data, sigma),
    }
)

parameter_values.update(
    {
        "Negative electrode active material volume fraction": pybop.Parameter(
            distribution=pybop.Gaussian(0.68, 0.02)
        ),
        "Positive electrode active material volume fraction": pybop.Parameter(
            distribution=pybop.Gaussian(0.65, 0.02)
        ),
    }
)

Build the problem and run the sampler to generate a result.

In [ ]:
simulator = pybop.pybamm.Simulator(
    model, parameter_values=parameter_values, protocol=dataset
)
cost = pybop.GaussianLogLikelihood(dataset)
log_pdf = pybop.LogPosterior(simulator, cost)

options = pybop.PintsSamplerOptions(
    n_chains=3,
    max_iterations=250,  # Extend this for accurate posteriors
    warm_up_iterations=100,
    verbose=True,
)
sampler = pybop.DifferentialEvolutionMCMC(log_pdf, options=options)
result = sampler.run()
result.get_summary_statistics();

## Choosing the plotting library

By default, plots are generated using matplotlib. We can use `pybop.plot.use_backend` to change the plotting library used by the plotting functions. Valid options are `'matplotlib'` and `'plotly'`.

Additionally, each plotting function takes an optional argument `backend` that overrides the current backend. This can either be a string (`'matplotlib'` or `'plotly'`) or an instance of `pybop.plot.backends.PlotBackend`. We will later see why the latter can be useful in very specific cases.

In [ ]:
result.plot_chains()  # matplotlib is default, may need to restart notebook to return to default

pybop.plot.use_backend("plotly")
pybop.plot.backends.PlotlyManager().pio.renderers.default = "notebook_connected"

result.plot_posterior()  # now using plotly
result.plot_posterior(backend="matplotlib")  # but still matplotlib for this plot
result.summary_table();  # and plotly for this one

## Combining multiple plots into one figure

_NOTE_:
When combining multiple plots into one figure, it is important to use the option `show = False` for all except the final plot. When using matplotlib, the plots will not work as expected without this. When using plotly, the plotting functions would still work as expected, except that unfinished versions of the figure would also be displayed.

### Subplots with plotly

First we generate empty subplots with plotly. We will later add a table to the plot, so we set the plot type as 'table' for the corresponding axis.

In [ ]:
from plotly.subplots import make_subplots

specs = [[{}] for _ in range(5)]
specs[4][0] = {"type": "table"}
fig = make_subplots(5, 1, specs=specs, horizontal_spacing=0.2, vertical_spacing=0.05)

We can now select the axes for each plot by passing the position of the axis as a tuple `(row, col)` to the plotting function. When using `matplotlib`, we would pass an Axis oject instead.

In [ ]:
result.plot_trace(figures=fig, axes=(1, 1), show=False)
result.plot_posterior(figures=fig, axes=(2, 1), show=False)
result.plot_chains(figures=fig, axes=(3, 1), show=False)
result.plot_predictive(figures=fig, axes=(4, 1), show=False);

_NOTE_: There seems to be a bug in plotly ([Issue #3424](https://github.com/plotly/plotly.py/issues/3424)).
If any plot with a vertical line is added to a figure with subplots AFTER a table was added, 
this may cause an error (vertical lines are added by plot_posterior and plot_chains, for example.)
This can be avoided by always adding any tables last.

In [ ]:
result.summary_table(figures=fig, axes=(5, 1), show=False)
fig.update_layout(height=1200)

## Customising a plot with matplotlib
To customise a plot with matplotlib after calling a `pybop.plot` plotting function, it is important to pass the argument `show=False` to avoid calling `plt.show()` prematurely. If `show=False` any plotting function will either return a single figure or a list of figures that can then be edited using standard `matplotlib` functionality. An even easier way to gain full control over the figure layout is to manually create the figure and pass it to the plotting function via the `figures` keyword argument.

By default, matplotlib cycles through the default colours separately for each axis in a figure. To use the same colorcycle across all axis in a figure, we can set the `global_colorcycle` property of the `pybop.plot.backends.MatplotlibBackend()` and pass the backend as an argument to all plotting functions that we want to share the same colorcycle.

In [ ]:
backend = pybop.plot.get_backend("matplotlib")
backend.global_colorcycle = True

We then generate a figure with matplotlib and pass the figure, its axes and the backend to the plotting function.

By default, `plot_trace` will create a legend for each axis. Now, if instead we want a legend for all axes, we can customise the output using standard `matplotlib` functionality. Since we used the option `show=False` when creating the plot, `plt.show()` has not been called yet and the figure is still available for updating.

In [ ]:
from matplotlib import pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(10, 8))
result.plot_trace(figures=fig, axes=axes, backend=backend, show=False)

handles = []
labels = []
for ax in axes:
    # remove individual legends
    ax.get_legend().remove()

    # get handles and labels of current axis
    hdls, lbls = ax.get_legend_handles_labels()

    # get the title of the current axis
    title = ax.get_title()

    # update labels to contain title of axis
    lbls = [
        pybop.plot.wrap_text(title + " - " + label, 20, "matplotlib") for label in lbls
    ]

    # add handles and updated labels to the global handles and labels list
    handles.extend(hdls)
    labels.extend(lbls)

# add a global legend
fig.legend(handles, labels)

# create some space for the legend
plt.tight_layout(rect=[0, 0, 0.8, 1])

# show the updated figure
plt.show();

## Using the SciencePlots package

When using the `matplotlib` backend we can easily use the SciencePlots package in combination with the plotting functions in PyBOP. Sometimes it might be necessary to manually generate the figure and pass it to the plotting function to properly apply the SciencePlots styling if a plotting function overrides any default options when generating the figure. This is not necessary in the following example.

In [ ]:
import scienceplots  # noqa: F401

pybop.plot.use_backend("matplotlib")

with plt.style.context(["science", "no-latex", "ieee"]):
    result.plot_trace()